# Content-Based Filtering

Predicts a user's rating on an unseen movie by aggregating their ratings on movies with **similar content**, where content is the concatenation `title + genres + user-supplied tags`. Two content representations are compared:

1. **TF-IDF** (sklearn) — sparse, fast, traditional baseline.
2. **Sentence embeddings** — `sentence-transformers/all-MiniLM-L6-v2`, a 384-dim dense embedding. Captures semantic relationships TF-IDF misses (e.g. *thriller* close to *suspense*).

For each (user, movie) test pair, we find the **top-30 most similar movies the user has rated** and predict their average rating, weighted by content similarity.

In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [2]:
!pip install -q sentence-transformers==2.7.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.1 MB/s eta 0:00:00


In [3]:
import os, random, sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

SEED = 42
random.seed(SEED); np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

ARTIFACT_DIR    = Path('/content/gdrive/MyDrive/recsys_artifacts')
SPLITS_DIR      = ARTIFACT_DIR / 'splits'
PREDICTIONS_DIR = ARTIFACT_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(ARTIFACT_DIR))
from recsys_metrics import build_candidate_lists, evaluate_model

In [4]:
train = pd.read_csv(SPLITS_DIR / 'train.csv')
val   = pd.read_csv(SPLITS_DIR / 'val.csv')
test  = pd.read_csv(SPLITS_DIR / 'test.csv')
trainval = pd.concat([train, val], ignore_index=True)

movies = pd.read_csv(ARTIFACT_DIR / 'movies_with_tags.csv')
movies = movies[['movieId', 'title', 'genres', 'all_tags', 'content_text']]
movies['content_text'] = movies['content_text'].fillna('').astype(str)
print(f'train: {len(train):,}  test: {len(test):,}  movies: {len(movies):,}')
movies.head(3)

train: 99,616  test: 610  movies: 9,742


,movieId,title,genres,all_tags,content_text
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,pixar pixar fun,Toy Story (1995) Adventure Animation Children ...
1,2,Jumanji (1995),Adventure|Children|Fantasy,fantasy magic board game robin williams game,Jumanji (1995) Adventure Children Fantasy fant...
2,3,Grumpier Old Men (1995),Comedy|Romance,moldy old,Grumpier Old Men (1995) Comedy Romance moldy old


## Variant A — TF-IDF on `title + genres + tags`

In [5]:
tfidf = TfidfVectorizer(min_df=2, ngram_range=(1, 2), stop_words='english')
X_tfidf = tfidf.fit_transform(movies['content_text'])
X_tfidf = normalize(X_tfidf, norm='l2', axis=1)  # so dot product = cosine similarity
print(f'TF-IDF matrix: {X_tfidf.shape}  nnz={X_tfidf.nnz}')

TF-IDF matrix: (9742, 6242)  nnz=81082


## Variant B — Sentence embeddings (`all-MiniLM-L6-v2`)

384-dim dense embeddings. Encoding the full catalogue takes ~30 seconds on CPU, a few seconds on a GPU runtime.

In [6]:
from sentence_transformers import SentenceTransformer
sbert = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X_sbert = sbert.encode(movies['content_text'].tolist(), batch_size=64, show_progress_bar=True, normalize_embeddings=True)
X_sbert = np.asarray(X_sbert, dtype=np.float32)
print(f'Sentence-transformer matrix: {X_sbert.shape}')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/153 [00:00<?, ?it/s]

Sentence-transformer matrix: (9742, 384)


## Prediction helper

Given an embedding matrix `X` (rows = movies in catalogue order), for each (user, movie) pair we look up the user's rated movies, compute cosine similarities to the candidate movie, take the top-30, and produce a similarity-weighted average rating.

In [7]:
movie_id_to_row = {m: i for i, m in enumerate(movies['movieId'].values)}
global_mean = float(trainval['rating'].mean())
TOP_K = 30

user_history = trainval.groupby('userId').apply(lambda d: list(zip(d['movieId'], d['rating']))).to_dict()

def make_predictor(X):
    """Closes over a content embedding matrix and returns (predict_pointwise, predict_fn)."""
    def predict_rating(user_id, movie_id):
        if movie_id not in movie_id_to_row or user_id not in user_history:
            return global_mean
        cand_row = movie_id_to_row[movie_id]
        history = user_history[user_id]
        rows = np.array([movie_id_to_row[mid] for mid, _ in history if mid in movie_id_to_row])
        ratings = np.array([r for mid, r in history if mid in movie_id_to_row], dtype=np.float32)
        if len(rows) == 0:
            return global_mean
        # Cosine similarity = dot product since vectors are L2-normalized.
        if hasattr(X, 'toarray'):  # sparse
            sims = np.asarray(X[rows] @ X[cand_row].T.toarray()).flatten()
        else:                     # dense
            sims = X[rows] @ X[cand_row]
        sims = np.clip(sims, 0, None)
        top = np.argsort(-sims)[:TOP_K]
        ws  = sims[top]
        rs  = ratings[top]
        if ws.sum() < 1e-9:
            return float(np.mean(ratings))
        return float(np.clip((ws * rs).sum() / ws.sum(), 0.5, 5.0))

    def predict_pointwise(df):
        out = df[['userId', 'movieId', 'rating']].copy()
        out['predicted_rating'] = [predict_rating(u, m) for u, m in zip(df['userId'], df['movieId'])]
        return out

    def predict_fn(user_id, movie_ids):
        return np.array([predict_rating(int(user_id), int(m)) for m in movie_ids], dtype=float)

    return predict_pointwise, predict_fn

/tmp/ipykernel_1206/978537227.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  user_history = trainval.groupby('userId').apply(lambda d: list(zip(d['movieId'], d['rating']))).to_dict()


## Evaluate both variants

In [8]:
candidates = build_candidate_lists(train, test, num_negatives=99, seed=SEED)

pw_tfidf, pr_tfidf = make_predictor(X_tfidf)
pw_sbert, pr_sbert = make_predictor(X_sbert)

pointwise_tfidf = pw_tfidf(test)
pointwise_sbert = pw_sbert(test)

metrics_tfidf = evaluate_model(pr_tfidf, test, candidates, pointwise_predictions=pointwise_tfidf, k=10, threshold=3.5)
metrics_sbert = evaluate_model(pr_sbert, test, candidates, pointwise_predictions=pointwise_sbert, k=10, threshold=3.5)

comparison = pd.DataFrame({
    'TF-IDF':              [metrics_tfidf[k] for k in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']],
    'SentenceTransformer': [metrics_sbert[k] for k in ['rmse', 'mae', 'hr_at_k', 'ndcg_at_k', 'precision_at_k', 'recall_at_k', 'f1_at_k']],
}, index=['RMSE', 'MAE', 'HR@10', 'NDCG@10', 'Precision@10', 'Recall@10', 'F1@10'])
comparison

,TF-IDF,SentenceTransformer
RMSE,0.986462,0.994450
MAE,0.747769,0.769038
HR@10,0.131148,0.155738
NDCG@10,0.062966,0.076409
Precision@10,0.011803,0.012459
Recall@10,0.118033,0.124590
F1@10,0.021461,0.022653


## Save the better variant's predictions

We pick whichever variant has lower test RMSE as `content_based.csv`. The sentence-transformer variant is also separately saved as `content_sbert.csv` so the hybrid GBRT notebook can use both as features if desired.

In [9]:
winner = 'sbert' if metrics_sbert['rmse'] < metrics_tfidf['rmse'] else 'tfidf'
pointwise_main = pointwise_sbert if winner == 'sbert' else pointwise_tfidf
pointwise_main[['userId', 'movieId', 'predicted_rating']].to_csv(PREDICTIONS_DIR / 'content_based.csv', index=False)
pointwise_sbert[['userId', 'movieId', 'predicted_rating']].to_csv(PREDICTIONS_DIR / 'content_sbert.csv', index=False)
print(f'Winner: {winner}')
print(f'Saved -> {PREDICTIONS_DIR / "content_based.csv"}')
print(f'Saved sentence-embedding variant -> {PREDICTIONS_DIR / "content_sbert.csv"}')

Winner: tfidf
Saved -> /content/gdrive/MyDrive/recsys_artifacts/predictions/content_based.csv
Saved sentence-embedding variant -> /content/gdrive/MyDrive/recsys_artifacts/predictions/content_sbert.csv
